# v3 rebuild — Stage 2 · subject-disjoint LOSO benchmark

Runs all eleven detectors plus the two-tower network under a strictly
subject-disjoint **Leave-One-Subject-Out** protocol on the Stage-1 clean features.
Answers Reviewer 3.2 / R1.5 / R3.6:

* each held-out subject is a fold (52 folds, not a subset);
* preprocessing (imputer + scaler) is refit **inside each training fold only**;
* features exclude the ID/Repetition leaks and duplicated EEG exports (Stage 1);
* per-fold macro-F1 / balanced-acc / accuracy are averaged over folds with a
  t-CI95; pooled (out-of-fold) confusion and per-class reports use raw counts;
* every model is compared against the best with a paired Wilcoxon signed-rank test;
* a stratified within-subject 80/20 run on the *same* features is kept as an explicit
  upper bound, never as the headline;
* per-window CPU inference latency is measured on a fitted mini-sample.

Writes under `v3_rebuild/benchmark/` (`fold_metrics_*.csv`, `benchmark_summary.csv`,
`oof_*.npz`, `paired_significance.csv`, `confusion_*_{best}.csv`, `per_class_{best}.csv`,
`within_subject_upper_bound.csv`, `inference_latency.csv`, `results.json`).

> **Runtime: ~2–4 h on commodity CPUs** (11 models × 52 folds; gradient boosting alone
> is ≈90 s/fold here). Run all cells only if you intend to recompute; the sealed
> outputs in `v3_rebuild/benchmark/` are the paper's source of truth and are
> byte-for-byte what this notebook reproduces.


In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split


def _root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "src" / "rebuild" / "models_registry.py").exists():
            return d
    raise SystemExit("run notebook from the module root or src/")
REPO = _root()
sys.path.insert(0, str(REPO / "src" / "rebuild"))

from models_registry import make_models, estimate_latency_us
OUT_ROOT = REPO / "output" / "research_outputs" / "fusion_training" / "v3_rebuild"
SEED = 42
print("out:", OUT_ROOT)

### Load the clean dataset + metric helpers

In [ ]:
from loso_benchmark import load_clean, fold_metrics, t_ci95, ece_from_1hot

df, X, y, le, subjects, feats = load_clean()
n_class = len(le.classes_)
print(f"rows={len(df)}  features={len(feats)}  subjects={len(np.unique(subjects))}  "
      f"classes={list(le.classes_)}")
print("class counts:", {k: int(v) for k, v in
                        pd.Series(y).map(lambda i: le.classes_[i]).value_counts().items()})

### Run the 52-fold LOSO loop for every model  (the long cell)

In [ ]:
bench_dir = OUT_ROOT / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)
registry = make_models(seed=SEED)
chosen = list(registry)
print("models:", chosen, flush=True)

summary_rows = []
for model_name in chosen:
    pipe = registry[model_name]
    fold_rows, oof_true, oof_pred, oof_proba = [], [], [], []
    for subj in sorted(np.unique(subjects)):
        tr, te = subjects != subj, subjects == subj
        pipe.fit(X[tr], y[tr])
        yp, yt = pipe.predict(X[te]), y[te]
        oof_true.append(yt)
        oof_pred.append(yp)
        if hasattr(pipe, "predict_proba"):
            oof_proba.append(pipe.predict_proba(X[te]))
        fold_rows.append({"model": model_name, "subject": int(subj), "n_test": int(te.sum()),
                          **fold_metrics(yt, yp, n_class)})
    folds = pd.DataFrame(fold_rows)
    folds.to_csv(bench_dir / f"fold_metrics_{model_name}.csv", index=False)

    oof_true = np.concatenate(oof_true)
    oof_pred = np.concatenate(oof_pred)
    proba = (np.concatenate(oof_proba) if oof_proba and len(oof_proba) == len(np.unique(subjects))
             else None)

    mf, ba, acc = folds.macro_f1, folds.balanced_acc, folds.accuracy
    lo_mf, hi_mf = t_ci95(mf)
    lo_ba, hi_ba = t_ci95(ba)
    summary_rows.append({
        "model": model_name, "folds": int(len(folds)),
        "macro_f1_mean": float(mf.mean()), "macro_f1_std": float(mf.std(ddof=1)),
        "macro_f1_ci95": [lo_mf, hi_mf],
        "balanced_acc_mean": float(ba.mean()), "balanced_acc_std": float(ba.std(ddof=1)),
        "balanced_acc_ci95": [lo_ba, hi_ba],
        "accuracy_mean": float(acc.mean()), "accuracy_std": float(acc.std(ddof=1)),
        "pooled_macro_f1": float(f1_score(oof_true, oof_pred, average="macro", zero_division=0)),
        "pooled_balanced_acc": float(balanced_accuracy_score(oof_true, oof_pred)),
        "pooled_accuracy": float(accuracy_score(oof_true, oof_pred)),
        "calib_ece": ece_from_1hot(oof_true, proba) if proba is not None else float("nan"),
    })
    np.savez_compressed(bench_dir / f"oof_{model_name}.npz",
                        y_true=oof_true, y_pred=oof_pred,
                        proba=proba if proba is not None else np.zeros((0, n_class)))
    print(f"[{model_name}] macro-F1 {mf.mean():.4f}±{mf.std(ddof=1):.4f}  "
          f"bal {ba.mean():.4f}  acc {acc.mean():.4f}  ECE {summary_rows[-1]['calib_ece']:.4f}",
          flush=True)

summary = (pd.DataFrame(summary_rows)
           .sort_values("macro_f1_mean", ascending=False)
           .reset_index(drop=True))
summary.insert(0, "rank", np.arange(1, len(summary) + 1))
summary.to_csv(bench_dir / "benchmark_summary.csv", index=False)
print("\n=== benchmark summary ===")
summary[["rank", "model", "macro_f1_mean", "macro_f1_std", "balanced_acc_mean",
         "accuracy_mean", "calib_ece"]]

### Paired significance vs the best (per-fold Wilcoxon over 52 folds)

In [ ]:
best = summary.iloc[0].model
sig_rows = []
for model_name in summary.model:
    if model_name == best:
        continue
    a = pd.read_csv(bench_dir / f"fold_metrics_{best}.csv").sort_values("subject").macro_f1
    b = pd.read_csv(bench_dir / f"fold_metrics_{model_name}.csv").sort_values("subject").macro_f1
    d = a.to_numpy() - b.to_numpy()
    try:
        w, p = stats.wilcoxon(d, zero_method="wilcox")
    except ValueError:
        w, p = float("nan"), float("nan")
    dz = d.mean() / (d.std(ddof=1) + 1e-12)
    sig_rows.append({"model": model_name, "vs": best,
                     "wilcoxon_p": float(p), "paired_dz": float(dz),
                     "delta_macro_f1": float(a.mean() - b.mean())})
sig = pd.DataFrame(sig_rows).sort_values("wilcoxon_p")
sig.to_csv(bench_dir / "paired_significance.csv", index=False)
print("best:", best)
sig

### Pooled per-class report + confusion matrix for the best model

In [ ]:
oof = np.load(bench_dir / f"oof_{best}.npz")
cm = confusion_matrix(oof["y_true"], oof["y_pred"], labels=np.arange(n_class))
pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv(
    bench_dir / f"confusion_counts_{best}.csv")
pd.DataFrame(cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1),
             index=le.classes_, columns=le.classes_).to_csv(
    bench_dir / f"confusion_normalized_{best}.csv")
rep = classification_report(oof["y_true"], oof["y_pred"], labels=np.arange(n_class),
                            target_names=le.classes_, output_dict=True, zero_division=0)
pd.DataFrame(rep).T.to_csv(bench_dir / f"per_class_{best}.csv")
pd.DataFrame(cm, index=le.classes_, columns=le.classes_)

### Within-subject upper bound (stratified 80/20, same features)

In [ ]:
ws_rows = []
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
for model_name in ["xgboost", "hist_gradient_boosting", "random_forest", "mlp"]:
    if model_name not in registry:
        continue
    p = registry[model_name]
    p.fit(Xtr, ytr)
    m = fold_metrics(yte, p.predict(Xte), n_class)
    ws_rows.append({"model": model_name, **m})
ws = pd.DataFrame(ws_rows).sort_values("macro_f1", ascending=False)
ws.to_csv(bench_dir / "within_subject_upper_bound.csv", index=False)
print("within-subject upper bound (stratified 80/20, clean features):")
ws

### Per-window CPU inference latency

In [ ]:
lat_rows = []
for model_name in summary.model:
    p = registry[model_name]
    idx = np.random.RandomState(0).choice(len(y), size=800, replace=False)
    p.fit(X[idx], y[idx])
    us = estimate_latency_us(p, X[:500])
    lat_rows.append({"model": model_name, "inference_us_per_window": us,
                     "windows_per_second": 1e6 / max(us, 1e-6)})
lat = pd.DataFrame(lat_rows).sort_values("inference_us_per_window")
lat.to_csv(bench_dir / "inference_latency.csv", index=False)
lat

### Seal + verify against the expected headline

In [ ]:
json.dump({"best_model": best, "summary": summary.to_dict(orient="records"),
           "classes": list(le.classes_)},
          open(bench_dir / "results.json", "w"), indent=2, default=float)

expect = dict(zip(summary.model, summary.macro_f1_mean))
print("best:", best, "| macro-F1", expect.get("gradient_boosting", float("nan")))
print("Expected (sealed, sklearn", "1.7.x", "+ xgboost): "
      "gradient_boosting 0.4661, xgboost 0.4620, hist_gradient_boosting 0.4511, "
      "random_forest 0.4300; within-subject ~0.91; gap ~0.45.")
summary[["rank", "model", "macro_f1_mean", "macro_f1_std", "calib_ece"]]